# Wan 2.1 Text-to-Video
🚀 ComfyUI + Cloudflare Tunnel (auto URL)

In [ ]:
# 1. Setup ComfyUI
%cd /content
!rm -rf ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI.git 2>&1 | tail -1
%cd ComfyUI
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt av 2>&1 | tail -1

In [ ]:
# 2. Install Wan nodes
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/kijai/ComfyUI-Wan.git 2>&1 | tail -1
print('Nodes installed')

In [ ]:
# 3. Download models
%cd /content/ComfyUI
!mkdir -p models/{unet,vae,clip}
!wget -q --show-progress -O models/unet/wan2.1_t2v_1.3B_bf16.safetensors \
  https://huggingface.co/Comfy-Org/Wan_2.1_T2V_1_3B-BP/resolve/main/split_files/diffusion_models/wan2.1_t2v_1.3B_bf16.safetensors
!wget -q --show-progress -O models/vae/wan_2.1_vae.safetensors \
  https://huggingface.co/Comfy-Org/Wan_2.1_T2V_1_3B-BP/resolve/main/split_files/vae/wan_2.1_vae.safetensors
!wget -q --show-progress -O models/clip/umt5_xxl_fp8_e4m3fn_scaled.safetensors \
  https://huggingface.co/Comfy-Org/Wan_2.1_T2V_1_3B-BP/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors
print('Models downloaded')

In [ ]:
# 4. Save Workflow
import json
workflow = {
 "last_node_id": 47,
 "last_link_id": 93,
 "nodes": [
 {"id": 38, "type": "CLIPLoader", "pos": [12, 184], "size": [390, 82], "widgets_values": ["umt5_xxl_fp8_e4m3fn_scaled.safetensors", "wan", "default"]},
 {"id": 37, "type": "UNETLoader", "pos": [485, 57], "size": [346, 82], "widgets_values": ["wan2.1_t2v_1.3B_bf16.safetensors", "default"]},
 {"id": 39, "type": "VAELoader", "pos": [866, 499], "size": [306, 58], "widgets_values": ["wan_2.1_vae.safetensors"]},
 {"id": 40, "type": "EmptyHunyuanLatentVideo", "pos": [520, 620], "size": [315, 130], "widgets_values": [832, 480, 33, 1]},
 {"id": 6, "type": "CLIPTextEncode", "pos": [415, 186], "size": [422, 164], "widgets_values": ["a fox moving quickly in a beautiful winter scenery nature trees sunset tracking camera"], "color": "#232", "bgcolor": "#353"},
 {"id": 7, "type": "CLIPTextEncode", "pos": [413, 389], "size": [425, 180], "widgets_values": ["Overexposure, static, blurred details, subtitles, paintings, pictures, still, overall gray, worst quality, low quality, JPEG compression residue, ugly, mutilated, redundant fingers, poorly painted hands, poorly painted faces, deformed, disfigured, deformed limbs, fused fingers, cluttered background, three legs, a lot of people in the background, upside down"], "color": "#322", "bgcolor": "#533"},
 {"id": 3, "type": "KSampler", "pos": [863, 187], "size": [315, 262], "widgets_values": [878361741413693, "randomize", 30, 6, "uni_pc", "simple", 1]},
 {"id": 8, "type": "VAEDecode", "pos": [1210, 190], "size": [210, 46]},
 {"id": 47, "type": "SaveWEBM", "pos": [2367, 193], "size": [315, 130], "widgets_values": ["ComfyUI", "vp9", 24, 32]},
 {"id": 28, "type": "SaveAnimatedWEBP", "pos": [1460, 190], "size": [870, 643], "widgets_values": ["ComfyUI", 16, False, 90, "default"]}
 ],
 "links": [
 [35, 3, 0, 8, 0, "LATENT"],
 [46, 6, 0, 3, 1, "CONDITIONING"],
 [52, 7, 0, 3, 2, "CONDITIONING"],
 [56, 8, 0, 28, 0, "IMAGE"],
 [74, 38, 0, 6, 0, "CLIP"],
 [75, 38, 0, 7, 0, "CLIP"],
 [76, 39, 0, 8, 1, "VAE"],
 [91, 40, 0, 3, 3, "LATENT"],
 [92, 37, 0, 3, 0, "MODEL"],
 [93, 8, 0, 47, 0, "IMAGE"]
 ]
}
with open('/content/ComfyUI/wan21_workflow.json', 'w') as f:
 json.dump(workflow, f, indent=2)
print('Workflow salvo em: /content/ComfyUI/wan21_workflow.json')

In [ ]:
# 5. START SERVER
import os, time
os.chdir('/content/ComfyUI')

# Kill old processes
!pkill -f 'main.py' 2>/dev/null; sleep 2

# Start
print('Starting ComfyUI...')
!nohup python main.py --listen 0.0.0.0 --port 8188 > /tmp/comfy.log 2>&1 &
time.sleep(20)

# Verify
!curl -s http://127.0.0.1:8188 | head -c 100
print('\nServer running! Now start tunnel:')
print('/proxy/8188')

In [ ]:
# 6. Cloudflare Tunnel (copy URL when ready)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!timeout 60 cloudflared tunnel --url http://localhost:8188 2>&1 | grep -E 'https://[a-z0-9]+\.trycloudflare\.com' || echo 'URL will appear above'